Quick and dirty notebook for simulation of coherent scattering data of magnetic samples in fraunhofer far-field regime

# Import

In [ ]:
# Import general libraries
import sys, os
from os.path import join, split
from importlib import reload
from copy import deepcopy
from tqdm.auto import tqdm

import numpy as np

# scipy
import scipy

# plotting
import matplotlib.pyplot as plt

# Interactive plotting
import ipywidgets
%matplotlib widget
plt.rcParams["figure.constrained_layout.use"] = True

In [ ]:
# Imports from our own codebase
from scattering_calculator.experimental_conditions import detector, light_beam
from scattering_calculator.sample_generator import pattern_generator
from scattering_calculator.sample_generator import structures
from scattering_calculator.beam_propagator import Jones_propagator, detector_effects
from scattering_calculator.utils import masking,physics,image_transformator
from scattering_calculator.interactive.interactive_widgets import cimshow


### EXPERIMENTAL GEOMETRY

In [ ]:
# ===================
# MATERIAL RECIPE
# ===================
recipe = "Au(1000)/Co(100)/Pt(2)"
#"Au(1000)/SiN(200)/Ta(5)/[Pt(1)/Co(1)]x15/Pt(3)"


#  ===================
# BASIC SAMPLE DIMENSION PARAMETERS
# ===================
sample_shape = np.array([0, 2**12, 2**12])  # in pixels
real_space_pixel_size = 2e-9  # in m


# ===================
# X-ray Source
# ===================
pol="CR"
x_ray_energy = 789.9  # eV
x_ray_photon_flux = 1e10  # Photons per pulse
beam_params = light_beam.beam_parameters(
    x_ray_energy, x_ray_photon_flux,pol
)

# Params for gaussian beam
illumination_function = "gaussian"
illumination_center = np.array(sample_shape[1:]) // 2  # in px
illumination_focus_distance = 0.#in m
illumination_fwhm = 5.50e-6  # in m


# ==================
# Geometry
# ==================
detector_pixel_size = 20e-6  # in m
detector_pixel_shape = (1300,1300)
detector_distance = 0.2  # in m
detector_center = (648,650)  # in px

# Optional: Define a beamstop
beamstop_radius = 0.5e-3  # in m
beamstop_distance = 0.001  # in m
beamstop_center = np.array(detector_pixel_shape) // 2  # in px

# ==================
# Setup
# ==================

# Basic camera parameters
exp_detector = detector.detector_layout(
    pixel_size=detector_pixel_size,
    detector_shape=detector_pixel_shape,
    distance_sample_detector=detector_distance,
    detector_center=detector_center
)


# Add beamstop to detector layout
beamstop = detector.beamstop(exp_detector, beamstop_distance)
beamstop.create_circle_beamstop(beamstop_center, beamstop_radius,use_real_space_coordinates=True)
bs_mask = beamstop.return_beamstop()
exp_detector.assign_beamstop(bs_mask)




exp_detector.calc_real_space_coordinates()
exp_detector.calc_q_space_coordinates(beam_params)


### SAMPLE STACK STRUCTURE AND OPTICAL PROPERTIES

In [ ]:

reload(structures)


# define stack
stack = structures.parse_recipe(
    recipe,
    sample_name="sample_A",
    comments=["test multilayer"],
)

#pulls material refractive indexes
material_params = structures.material_params(materials=set([element.material for element in stack.layers]), x_ray_energy=x_ray_energy)


sample = structures.Structure(
    name="Test Structure",
    material_params=material_params,
    sample_shape=sample_shape,
    real_space_pixel_size=real_space_pixel_size
)

for layer in stack.layers:
    sample.add_layer(layer.material, thickness=layer.thickness)



# SAMPLE 

### - magnetic domains

In [ ]:

reload(pattern_generator)

%time
# Skyrmion and Screening diameter
skyrmion_radius = 40e-9  # m
screening_radius = (
    1.25 * skyrmion_radius
)  # m, either only even or odd, otherwise script will fail
skyrmion_smoothing = 1

# Number of skyrmions
# If this number is too high, the script may take forever ...
# (brute force algorithm)
number_of_skyrmions = 600000
max_nr_iteration = 3000  # 10 * sample["no_skyr"]

# Create Pattern
skyrmion_pattern, coordinates = pattern_generator.create_skyrmion_pattern(
    sample.sample_shape[1:],
    skyrmion_radius/sample.real_space_pixel_size,
    screening_radius/sample.real_space_pixel_size,
    number_of_skyrmions,
    max_nr_iteration,
    sigma=skyrmion_smoothing,
    real_space_pixel_size = sample.real_space_pixel_size,
    plot=True,
)

sample.magnetization=pattern_generator.map_magnetization_to_3d(0*skyrmion_pattern,np.sqrt(1-np.abs(skyrmion_pattern)**2),skyrmion_pattern,nr_repeats=sample_shape[0])



# - holography mask

In [ ]:
reload(structures)
reload(masking)


front_aperture = structures.Apertures3D(sample_shape, real_space_pixel_size,
    layer_thicknesses=sample.layer_thicknesses)

if True:
    front_aperture_radius = 500e-9  # in m
    front_aperture.create_circle_aperture(
        center=(sample.sample_shape[1] // 2, sample.sample_shape[2] // 2),
        depth=1000e-9,
        radius=front_aperture_radius,
        use_real_space_coordinates=True,
        sigma=10e-9
    )


front_aperture_radius = 30e-9  # in m
front_aperture.create_circle_aperture(
    center=(sample.sample_shape[1] // 2 + 850, sample.sample_shape[2] // 2 + 860),
    depth=np.sum(sample.layer_thicknesses),
    radius=front_aperture_radius,
    use_real_space_coordinates=True,
    sigma=5e-9
)


sample.mask=front_aperture.aperture_design

front_aperture.visualize_aperture()


### get dielectric tensor

In [ ]:

sample.calculate_final_dielectric_tensor()

fig,ax=plt.subplots()
ax.imshow( np.sum(np.abs(sample.final_dielectric_tensor), axis=(0,3,4)))

### HOLOGRAM COMPUTATION

In [ ]:
import time

reload(Jones_propagator)
reload(light_beam)
reload(detector_effects)
reload(detector)

holos=[None,None]
ii=0

time0=time.time()
for beam_params.pol in ["CR", "CL"]:
    illumination = light_beam.illumination(beam_params, sample_shape[1:], real_space_pixel_size)

    # Comment: Check gauss_beam function for different focus distances
    illumination.gauss_beam(
        illumination_center,
        illumination_focus_distance,
        illumination_fwhm,
    )
    illumination.get_illumination_jones()


    illumination_wavefield = illumination.return_illumination()
    extent_illumination_real = illumination.get_illumination_extent_real_space()
    #illumination.visualize_illumination()


    print("illwave",time.time()-time0)
    time0=time.time()

    ### Do light propagation

    reload(Jones_propagator)
    reload(detector_effects)

    wavefront = Jones_propagator.wavefronts(beam_parameters=beam_params,
                                            eps_stack=sample.final_dielectric_tensor,
                                            layer_thicknesses=sample.layer_thicknesses,
                                            real_space_pixel_size=sample.real_space_pixel_size,
                                            E_in=illumination.illumination_jones,
                                            propagate=False)


    print("jones_prop",time.time()-time0)
    time0=time.time()

    hologram_exp = detector_effects.detector_hologram( exp_detector, wavefront.hologram, beam_params,sample.real_space_pixel_size, beamstop)
    hologram_exp.gnomonic_projection()
    hologram_exp.add_noise()

    print("gnomonic",time.time()-time0)
    time0=time.time()

    holos[ii]=hologram_exp.hologram_exp*(1-beamstop.beamstop)
    ii+=1

In [ ]:
reload(detector_effects)
hologram_exp.add_noise()
cimshow(hologram_exp.hologram_exp)

In [ ]:
plt.close("all")


fig,ax=plt.subplots(2,2)

roi=np.s_[200:500,200:500]
fth=Jones_propagator.reconstruct((holos[0]+holos[1]))
ax[0,0].imshow(np.log10(holos[0]+holos[1]))
ax[0,1].imshow((np.real(fth)[roi]))

fth=Jones_propagator.reconstruct(holos[0]-holos[1])
ax[1,0].imshow(np.log10(np.abs(holos[0]-holos[1])))
ax[1,1].imshow((np.real(fth)[roi]))

